In [ ]:
print("hello world")

In [ ]:
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.layers import Dense ,Dropout , Flatten , Conv2D , MaxPooling2D
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorlow.keras.utils import plot_model
import matplotlib.pyplot as plt

from sklearn.preprocessing import train_test_split

In [ ]:
train_ds = image_dataset_from_directory(
    directory = "/home/aman/Desktop/Kidney-Disease-classifier/Data/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone",
    labels = "inferred",
    label_mode = "categorical",
    image_size = (224,224),
    batch_size = 32,
    color_mode = "grayscale",
    shuffle = True,   
    validation_split = 0.2,
    subset = "Training",
    seed = 123 
)

In [ ]:
train_ds , valid_ds = train_test_split(train_ds, test_size=0.2, random_state=42)

In [ ]:
test_ds = image_dataset_from_directory(
    directory = "/home/aman/Desktop/Kidney-Disease-classifier/Data/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone",
    labels = "inferred",
    label_mode = "categorical",
    image_size = (224,224),
    batch_size = 32,
    color_mode = "grayscale",
    shuffle = True,   
    validation_split = 0.2,
    subset = "validation",
    seed = 123 
)

In [ ]:
model = Sequential()
model.add(Flatten(input_shape=(224,224,1)))
model.add(Dense(256, activation='relu'))
model.add(Dense(128, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(4, activation='softmax'))

In [ ]:
def final_model(hp):
    model = Sequential()

    nums_layers = hp.Int('num_layers', min_value=1, max_value=10, step=1)
    for i in range(nums_layers):
        if i == 0:
            model.add(Flatten(input_shape=(224,224,1)))
        else:
            model.add(Dense(
                units=hp.Int('units_' + str(i), min_value=32, max_value=512, step=32),
                activation = hp.Choice('activation_' + str(i), values=['relu', 'tanh', 'sigmoid'])
            ))

            model.add(Dropout(rate=hp.Float('dropout_' + str(i), min_value=0.0, max_value=0.5, step=0.1)))

    model.add(Dense(4, activation='softmax'))

    model.compile(
        optimizer = hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop','Nadam']),
        loss = 'categorical_crossentropy',
        metrics = ['accuracy']
    )

    return model




In [ ]:
tuner = kt.Hyperband(
    final_model,
    objective = 'val_accuracy',
    max_trials = 10,
    directory = 'hyperband',
    project_name = 'kidney_disease_classifier'
)

In [ ]:
tuner.search(training_data = train_ds, epochs=10, validation_data=valid_ds)

In [ ]:
model = tuner.get_best_models(num_models=1)[0]

In [ ]:
model.summary()

In [ ]:
early_stopping = EarlyStopping(monitor='val_loss', patience=5,verbose=1, mode='min',restore_best_weights=True)



In [ ]:
history = model.fit(
    train_ds,
    validation_data = test_ds,
    epochs = 50,
    callbacks = [early_stopping]
)

In [ ]:
plot_model(model, to_file='model.png', show_shapes=True, show_layer_names=True)

In [ ]:
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()